In [ ]:
%%shell
# 1. Download official Google Chrome signing key
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -

# 2. Add Google's repository
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list

# 3. Update and install Chrome
apt-get update -y
apt-get install -y google-chrome-stable

# 4. Install Selenium and the brilliant Webdriver Manager
pip install selenium webdriver-manager lxml

OK
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,213 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,793 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 https://r2u.stat.illinois.edu/ubuntu 

# Rehap and Recovery data

# exercise name with url that will we scraped

In [ ]:
urls_to_scan = {
    # The Muscle Tears (Sprinting & Kicking)
    "Hamstring Strain": "https://www.physio-pedia.com/Hamstring_Strain",
    "Groin Strain": "https://www.physio-pedia.com/Groin_Strain",
    "Calf Strain": "https://www.physio-pedia.com/Calf_Strain",
    "Hip Flexor Strain": "https://www.physio-pedia.com/Iliopsoas_Tendinopathy",

    # The Impact Injuries (Tackles & Falls)
    "Quadriceps Contusion": "https://www.physio-pedia.com/Quadriceps_Muscle_Contusion",
    "Metatarsal Fracture": "https://www.physio-pedia.com/Metatarsal_Fractures",
    "Concussion": "https://www.physio-pedia.com/Concussion",

    # The Knee Joint (Pivoting & Turf)
    "ACL Injury": "https://www.physio-pedia.com/Anterior_Cruciate_Ligament_(ACL)_Injury",
    "Meniscus Lesion": "https://www.physio-pedia.com/Meniscal_Lesions",
    "MCL Sprain": "https://www.physio-pedia.com/Medial_Collateral_Ligament_Injury_of_the_Knee",
    "Patellar Tendinopathy": "https://www.physio-pedia.com/Patellar_Tendinopathy",
    "IT Band Syndrome": "https://www.physio-pedia.com/Iliotibial_Band_Syndrome",

    # The Ankle, Foot & Core (Ground Contact)
    "Lateral Ankle Sprain": "https://www.physio-pedia.com/Ankle_Sprain",
    "Ankle Syndesmosis Injuries": "https://www.physio-pedia.com/Ankle_Syndesmosis_Injuries",
    "Achilles Tendinopathy": "https://www.physio-pedia.com/Achilles_Tendinopathy",
    "Sports Hernia": "https://www.physio-pedia.com/Pubalgia"
}
print(len(urls_to_scan.keys()))

16


## scrap header and title scrap all title

In [ ]:
import time
import json
from lxml import html
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# 1. SETUP INVISIBLE CHROME
# ==========================================
print("Spinning up invisible Chrome browser...")
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# ==========================================
# 2. THE TARGET URLS
# ==========================================
urls_to_scan = {
    # The Muscle Tears (Sprinting & Kicking)
    "Hamstring Strain": "https://www.physio-pedia.com/Hamstring_Strain",
    "Groin Strain": "https://www.physio-pedia.com/Groin_Strain",
    "Calf Strain": "https://www.physio-pedia.com/Calf_Strain",
    "Hip Flexor Strain": "https://www.physio-pedia.com/Iliopsoas_Tendinopathy",

    # The Impact Injuries (Tackles & Falls)
    "Quadriceps Contusion": "https://www.physio-pedia.com/Quadriceps_Muscle_Contusion",
    "Metatarsal Fracture": "https://www.physio-pedia.com/Metatarsal_Fractures",
    "Concussion": "https://www.physio-pedia.com/Concussion",

    # The Knee Joint (Pivoting & Turf)
    "ACL Injury": "https://www.physio-pedia.com/Anterior_Cruciate_Ligament_(ACL)_Injury",
    "Meniscus Lesion": "https://www.physio-pedia.com/Meniscal_Lesions",
    "MCL Sprain": "https://www.physio-pedia.com/Medial_Collateral_Ligament_Injury_of_the_Knee",
    "Patellar Tendinopathy": "https://www.physio-pedia.com/Patellar_Tendinopathy",
    "IT Band Syndrome": "https://www.physio-pedia.com/Iliotibial_Band_Syndrome",

    # The Ankle, Foot & Core (Ground Contact)
    "Lateral Ankle Sprain": "https://www.physio-pedia.com/Ankle_Sprain",
    "Ankle Syndesmosis Injuries": "https://www.physio-pedia.com/Ankle_Syndesmosis_Injuries",
    "Achilles Tendinopathy": "https://www.physio-pedia.com/Achilles_Tendinopathy",
    "Sports Hernia": "https://www.physio-pedia.com/Pubalgia"
}

page_structures = {}
all_unique_headers = set()
failed_urls = []  # Track failed URLs

# ==========================================
# 3. THE PROFILING LOOP WITH SUCCESS/FAILURE TRACKING
# ==========================================
print("\n🔍 Starting Header Profiling...\n")

for name, url in urls_to_scan.items():
    print(f"\n{'='*50}")
    print(f"📄 Processing: {name}")
    print(f"🔗 URL: {url}")
    print(f"{'='*50}")

    try:
        driver.get(url)
        time.sleep(3) # Wait for Cloudflare

        # Check if page loaded successfully
        if "404" in driver.title or "Page not found" in driver.page_source:
            print(f"❌ FAILED: Page not found (404)")
            failed_urls.append({"name": name, "url": url, "reason": "404 Not Found"})
            continue

        tree = html.fromstring(driver.page_source)

        # Check if main content exists
        content_div = tree.xpath("//div[@id='mw-content-text']")
        if not content_div:
            print(f"❌ FAILED: Could not find main content div")
            failed_urls.append({"name": name, "url": url, "reason": "No content div"})
            continue

        # Grab all h2, h3, and h4 tags inside the main content area
        all_headers = tree.xpath("//div[@id='mw-content-text']//h2 | //div[@id='mw-content-text']//h3 | //div[@id='mw-content-text']//h4")

        if not all_headers:
            print(f"⚠️ WARNING: No headers found on page")
            extracted_headers = []
        else:
            extracted_headers = []

            for header in all_headers:
                text = header.text_content().strip()

                # Skip empty headers
                if not text:
                    continue

                # Break point logic
                lower_text = text.lower()
                if "reference" in lower_text or "related article" in lower_text or "see also" in lower_text:
                    print(f"  🛑 Break point hit at: '{text}'")
                    break

                # Clean up the text
                clean_text = text.replace("[edit | edit source]", "").strip()

                extracted_headers.append(clean_text)
                all_unique_headers.add(clean_text)

        page_structures[name] = extracted_headers

        # Print success summary
        print(f"✅ SUCCESS: Found {len(extracted_headers)} headers")
        if extracted_headers:
            print(f"   First 3 headers: {extracted_headers[:3]}")
            if len(extracted_headers) > 3:
                print(f"   ... and {len(extracted_headers)-3} more")

    except Exception as e:
        print(f"❌ FAILED: Error - {str(e)}")
        failed_urls.append({"name": name, "url": url, "reason": str(e)})
        page_structures[name] = []  # Store empty list for failed pages

driver.quit()

# ==========================================
# 4. DISPLAY DETAILED RESULTS
# ==========================================
print("\n" + "="*60)
print("🎯 PROFILING COMPLETE - DETAILED REPORT")
print("="*60)

# Summary statistics
total_urls = len(urls_to_scan)
successful = len([p for p in page_structures.values() if p])
failed = len(failed_urls)

print(f"\n📊 SUMMARY STATISTICS:")
print(f"   • Total URLs: {total_urls}")
print(f"   • ✅ Successful: {successful}")
print(f"   • ❌ Failed: {failed}")
print(f"   • 📝 Total unique headers found: {len(all_unique_headers)}")

# Failed URLs detailed report
if failed_urls:
    print(f"\n❌ FAILED URLS REPORT:")
    print("-" * 40)
    for i, fail in enumerate(failed_urls, 1):
        print(f"{i}. {fail['name']}")
        print(f"   URL: {fail['url']}")
        print(f"   Reason: {fail['reason']}")
        print()

# Successful pages detailed headers
print(f"\n✅ SUCCESSFUL PAGES - HEADER DETAILS:")
print("-" * 40)
for name, headers in page_structures.items():
    if headers:  # Only show successful pages
        print(f"\n📄 {name}:")
        print(f"   Total headers: {len(headers)}")
        print("   Headers found:")
        for i, header in enumerate(headers, 1):
            print(f"     {i}. {header}")

# Show what was common vs unique
print(f"\n🧠 HEADER ANALYSIS:")
print("-" * 40)
print(f"📋 MASTER LIST OF ALL UNIQUE HEADERS ({len(all_unique_headers)} total):")
print()
for i, header in enumerate(sorted(list(all_unique_headers)), 1):
    # Count how many pages have this header
    pages_with_header = sum(1 for headers in page_structures.values() if header in headers)
    popularity = "🔥" if pages_with_header > len(page_structures)/2 else "📌"
    print(f"   {popularity} {header:<40} (found in {pages_with_header}/{len(page_structures)} pages)")

# ==========================================
# 5. SAVE EVERYTHING TO ONE FILE
# ==========================================
output_data = {
    "scan_summary": {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_urls": total_urls,
        "successful": successful,
        "failed": failed,
        "unique_headers_total": len(all_unique_headers)
    },
    "failed_urls": failed_urls,
    "page_structures": page_structures,
    "unique_headers": sorted(list(all_unique_headers)),
    "header_popularity": {
        header: sum(1 for headers in page_structures.values() if header in headers)
        for header in all_unique_headers
    }
}

# Save to file
with open('header_profile.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"\n💾 Saved complete report to 'header_profile.json'")

# Quick check of what was saved
print(f"\n🔍 QUICK CHECK:")
print(f"   • File contains: {len(output_data['page_structures'])} page structures")
print(f"   • Failed URLs logged: {len(output_data['failed_urls'])}")
print(f"   • Unique headers cataloged: {len(output_data['unique_headers'])}")

Spinning up invisible Chrome browser...

🔍 Starting Header Profiling...


📄 Processing: Hamstring Strain
🔗 URL: https://www.physio-pedia.com/Hamstring_Strain
  🛑 Break point hit at: 'References[edit | edit source]'
✅ SUCCESS: Found 18 headers
   First 3 headers: ['Contents', 'Introduction', 'Epidemiology/Etiology']
   ... and 15 more

📄 Processing: Groin Strain
🔗 URL: https://www.physio-pedia.com/Groin_Strain
❌ FAILED: Could not find main content div

📄 Processing: Adductor Tendinopathy
🔗 URL: https://www.physio-pedia.com/Adductor_Tendinopathy
❌ FAILED: Could not find main content div

📄 Processing: Calf Strain
🔗 URL: https://www.physio-pedia.com/Calf_Strain
❌ FAILED: Could not find main content div

📄 Processing: Rectus Femoris Strain
🔗 URL: https://www.physio-pedia.com/Rectus_Femoris


KeyboardInterrupt: 

# scrap the content and explanition of header in each exercise

In [ ]:
import time
import json
from lxml import html
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# 1. SETUP INVISIBLE CHROME BROWSER
# ==========================================
print("Spinning up invisible Chrome browser...\n")
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)


urls_to_scan = {
    # The Muscle Tears (Sprinting & Kicking)
    "Hamstring Strain": "https://www.physio-pedia.com/Hamstring_Strain",
    "Groin Strain": "https://www.physio-pedia.com/Groin_Strain",
    "Calf Strain": "https://www.physio-pedia.com/Calf_Strain",
    "Hip Flexor Strain": "https://www.physio-pedia.com/Iliopsoas_Tendinopathy",

    # The Impact Injuries (Tackles & Falls)
    "Quadriceps Contusion": "https://www.physio-pedia.com/Quadriceps_Muscle_Contusion",
    "Metatarsal Fracture": "https://www.physio-pedia.com/Metatarsal_Fractures",
    "Concussion": "https://www.physio-pedia.com/Concussion",

    # The Knee Joint (Pivoting & Turf)
    "ACL Injury": "https://www.physio-pedia.com/Anterior_Cruciate_Ligament_(ACL)_Injury",
    "Meniscus Lesion": "https://www.physio-pedia.com/Meniscal_Lesions",
    "MCL Sprain": "https://www.physio-pedia.com/Medial_Collateral_Ligament_Injury_of_the_Knee",
    "Patellar Tendinopathy": "https://www.physio-pedia.com/Patellar_Tendinopathy",
    "IT Band Syndrome": "https://www.physio-pedia.com/Iliotibial_Band_Syndrome",

    # The Ankle, Foot & Core (Ground Contact)
    "Lateral Ankle Sprain": "https://www.physio-pedia.com/Ankle_Sprain",
    "Ankle Syndesmosis Injuries": "https://www.physio-pedia.com/Ankle_Syndesmosis_Injuries",
    "Achilles Tendinopathy": "https://www.physio-pedia.com/Achilles_Tendinopathy",
    "Sports Hernia": "https://www.physio-pedia.com/Pubalgia"
}

master_dynamic_database = []

# ==========================================
# 3. THE DYNAMIC EXTRACTION ENGINE
# ==========================================
for injury_name, url in urls_to_scan.items():
    print("="*60)
    print(f"🚀 Scraping: {injury_name}")
    print("="*60)

    driver.get(url)
    time.sleep(3) # Wait for Cloudflare/JS
    tree = html.fromstring(driver.page_source)

    injury_data = {
        "injury_name": injury_name,
        "source_url": url,
        "sections": {}
    }

    # Step A: Get Introduction text (before the first header starts)
    intro_content = []
    first_paragraphs = tree.xpath("(//div[@id='mw-content-text']//p)[position() <= 3]")
    for p in first_paragraphs:
        text = p.text_content().strip()
        if text: intro_content.append({"text": " ".join(text.split())})

    if intro_content:
        injury_data["sections"]["Introduction_Introductory_Text"] = intro_content
        print(f"  ✅ SUCCESS: Introduction_Introductory_Text ({len(intro_content)} blocks)")

    # Step B: Dynamically find ALL headers in document order
    all_headers = tree.xpath("//div[@id='mw-content-text']//h2 | //div[@id='mw-content-text']//h3 | //div[@id='mw-content-text']//h4")

    for header in all_headers:
        raw_title = header.text_content().strip()

        # Clean the title of Wiki artifacts
        clean_title = raw_title.replace("[edit | edit source]", "").strip()

        if not clean_title:
            continue

        # ✨ THE CIRCUIT BREAKER ✨
        lower_title = clean_title.lower()
        if "reference" in lower_title or "related article" in lower_title or "see also" in lower_title:
            print(f"  🛑 BREAK POINT HIT: '{clean_title}'. Moving to next injury.\n")
            break

        # Step C: Extract all siblings until the next header
        section_content = []
        for elem in header.itersiblings():
            # Stop grabbing when we hit the next header
            if elem.tag in ['h2', 'h3', 'h4']:
                break

            # 1. Grab Text Paragraphs
            text = elem.text_content().strip()
            if text and elem.tag not in ['ul', 'table']:
                section_content.append({"text": " ".join(text.split())})

            # 2. Grab Bulleted Lists
            if elem.tag == 'ul':
                items = [" ".join(li.text_content().split()) for li in elem.xpath(".//li")]
                if items: section_content.append({"list": items})

            # 3. Grab Tables
            tables = elem.xpath(".//table") if elem.tag != 'table' else [elem]
            for table in tables:
                rows = []
                headers_cells = table.xpath(".//tr/th")
                header_row = [h.text_content().strip() for h in headers_cells] if headers_cells else None
                for tr in table.xpath(".//tr"):
                    cells = [c.text_content().strip() for c in tr.xpath(".//th|.//td")]
                    if cells: rows.append(cells)
                if rows: section_content.append({"table": {"header": header_row, "rows": rows}})

        # Step D: Enhanced Logging & Saving
        if section_content:
            injury_data["sections"][clean_title] = section_content
            print(f"  ✅ SUCCESS: {clean_title} ({len(section_content)} items)")
        else:
            print(f"  ⚠️ EMPTY:   {clean_title} (No immediate content found)")

    master_dynamic_database.append(injury_data)

driver.quit()

# ==========================================
# 4. SAVE THE DYNAMIC DATABASE
# ==========================================
with open('dynamic_master_database.json', 'w', encoding='utf-8') as f:
    json.dump(master_dynamic_database, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print(f"🎉 SCRAPING COMPLETE! Saved to 'dynamic_master_database.json'")
print("="*60)

Starting stealth Chrome browser...



SessionNotCreatedException: Message: session not created: cannot connect to chrome at 127.0.0.1:35203
from chrome not reachable; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x56870fd2ceea <unknown>
#1 0x56870f73b8f0 <unknown>
#2 0x56870f72700c <unknown>
#3 0x56870f77d9de <unknown>
#4 0x56870f772bf1 <unknown>
#5 0x56870f7c357e <unknown>
#6 0x56870f7c2c6c <unknown>
#7 0x56870f781cef <unknown>
#8 0x56870f782ab1 <unknown>
#9 0x56870fcf1d64 <unknown>
#10 0x56870fcf4c51 <unknown>
#11 0x56870fcdeb6a <unknown>
#12 0x56870fcf581e <unknown>
#13 0x56870fcc56e0 <unknown>
#14 0x56870fd19b18 <unknown>
#15 0x56870fd19ce8 <unknown>
#16 0x56870fd2b92e <unknown>
#17 0x78987c7baac3 <unknown>


# use regex to remove un needed char like[number] and apply basic text cleansing

In [ ]:
import json
import re

# ==========================================
# 1. LOAD THE RAW DYNAMIC DATABASE
# ==========================================
try:
    with open('dynamic_master_database.json', 'r', encoding='utf-8') as f:
        master_database = json.load(f)
    print(f"✅ Loaded {len(master_database)} injuries for cleaning.\n")
except FileNotFoundError:
    print("❌ Could not find 'dynamic_master_database.json'. Please run the scraper first.")
    master_database = []

# ==========================================
# 2. THE CLEANING FUNCTIONS
# ==========================================
def clean_text(text):
    """Uses Regex to remove citations like [1], [12], [1, 2], [1-3] and trims spaces."""
    if not isinstance(text, str):
        return text

    # Regex explanation:
    # \[     : Match opening bracket
    # \d+    : Match one or more numbers
    # (?:[-,\s]\d+)* : Match things like ", 2" or "-3" if they exist inside the brackets
    # \]     : Match closing bracket
    cleaned = re.sub(r'\[\d+(?:[-,\s]\d+)*\]', '', text)

    # Remove any weird double spaces created by the deletion
    cleaned = " ".join(cleaned.split())
    return cleaned

def clean_content_blocks(blocks):
    """Walks through the text, lists, and tables to clean every single string."""
    cleaned_blocks = []

    for block in blocks:
        # A. Clean standard text paragraphs
        if 'text' in block:
            txt = clean_text(block['text'])
            if txt:  # Only keep if it's not empty
                cleaned_blocks.append({'text': txt})

        # B. Clean bulleted lists
        elif 'list' in block:
            cleaned_list = [clean_text(item) for item in block['list']]
            # Remove empty list items
            cleaned_list = [item for item in cleaned_list if item]
            if cleaned_list:
                cleaned_blocks.append({'list': cleaned_list})

        # C. Clean Table Headers and Rows
        elif 'table' in block:
            headers = block['table'].get('header')
            if headers:
                headers = [clean_text(h) for h in headers]

            rows = block['table'].get('rows', [])
            cleaned_rows = []
            for row in rows:
                cleaned_row = [clean_text(cell) for cell in row]
                # Only keep the row if it actually contains text
                if any(cleaned_row):
                    cleaned_rows.append(cleaned_row)

            if headers or cleaned_rows:
                cleaned_blocks.append({'table': {'header': headers, 'rows': cleaned_rows}})

    return cleaned_blocks

# ==========================================
# 3. THE PRUNING LOOP
# ==========================================
print("🧹 Starting Regex Cleaning and Empty Section Pruning...\n")

total_citations_removed = 0
total_sections_removed = 0

for injury in master_database:
    injury_name = injury.get("injury_name", "Unknown")
    sections = injury.get("sections", {})

    keys_to_delete = []

    for section_name, blocks in sections.items():
        # 1. Clean the text inside the blocks
        original_length = len(str(blocks))
        cleaned_blocks = clean_content_blocks(blocks)
        new_length = len(str(cleaned_blocks))

        # 2. Check if the block is now totally empty
        if not cleaned_blocks:
            keys_to_delete.append(section_name)
        else:
            injury["sections"][section_name] = cleaned_blocks

    # 3. Delete the empty sections from the dictionary
    for key in keys_to_delete:
        del injury["sections"][key]
        total_sections_removed += 1

    print(f"  ✅ {injury_name}: Pruned {len(keys_to_delete)} empty sections.")

# ==========================================
# 4. SAVE THE SANITIZED DATABASE
# ==========================================
with open('sanitized_master_database.json', 'w', encoding='utf-8') as f:
    json.dump(master_database, f, indent=4, ensure_ascii=False)

print("\n" + "="*50)
print(f"🎉 SANITIZATION COMPLETE!")
print(f"🗑️ Total Empty Sections Deleted: {total_sections_removed}")
print("💾 Saved to 'sanitized_master_database.json'")
print("="*50)

✅ Loaded 16 injuries for cleaning.

🧹 Starting Regex Cleaning and Empty Section Pruning...

  ✅ Hamstring Strain: Pruned 1 empty sections.
  ✅ Groin Strain: Pruned 1 empty sections.
  ✅ Calf Strain: Pruned 0 empty sections.
  ✅ Hip Flexor Strain: Pruned 0 empty sections.
  ✅ Quadriceps Contusion: Pruned 0 empty sections.
  ✅ Metatarsal Fracture: Pruned 0 empty sections.
  ✅ Concussion: Pruned 0 empty sections.
  ✅ ACL Injury: Pruned 0 empty sections.
  ✅ Meniscus Lesion: Pruned 0 empty sections.
  ✅ MCL Sprain: Pruned 0 empty sections.
  ✅ Patellar Tendinopathy: Pruned 0 empty sections.
  ✅ IT Band Syndrome: Pruned 0 empty sections.
  ✅ Lateral Ankle Sprain: Pruned 0 empty sections.
  ✅ Ankle Syndesmosis Injuries: Pruned 0 empty sections.
  ✅ Achilles Tendinopathy: Pruned 0 empty sections.
  ✅ Sports Hernia: Pruned 0 empty sections.

🎉 SANITIZATION COMPLETE!
🗑️ Total Empty Sections Deleted: 2
💾 Saved to 'sanitized_master_database.json'


# remove all unneeded sector or parent for adds or article (trash data)

In [ ]:
import json

# ==========================================
# 1. LOAD THE SANITIZED DATABASE
# ==========================================
try:
    with open('sanitized_master_database.json', 'r', encoding='utf-8') as f:
        master_database = json.load(f)
    print(f"✅ Loaded {len(master_database)} injuries for targeted pruning.\n")
except FileNotFoundError:
    print("❌ Could not find 'sanitized_master_database.json'.")
    master_database = []

# ==========================================
# 2. THE "POISON WORD" DICTIONARY
# ==========================================
# I changed the capital 'R' to lowercase 'r' here
JUNK_TITLES = [
    "pai",
    "ai clinical assistant",
    "700+",
    "accredited online courses",
    "presentations",
    "introduction_introductory_text",
    "viewing",
    "resources"
]

# I changed the capital 'R' to lowercase 'r' here
JUNK_CONTENT = [
    "your ai clinical assistant",
    "plus online learning",
    "get your ceus / cpd points from physiopedia",
    "ai supported learning powered by physiopedia",
    "professional development from physiopedia plus"
]

# ==========================================
# 3. THE PRUNING LOOP
# ==========================================
print("🧹 Hunting for Ads and Footer Noise...\n")

total_deleted = 0

for injury in master_database:
    injury_name = injury.get("injury_name", "Unknown")
    sections = injury.get("sections", {})

    keys_to_delete = []

    # Check every section in this injury
    for section_title, blocks in sections.items():
        title_lower = section_title.lower()
        content_string = json.dumps(blocks).lower()

        delete_this_section = False
        reason = ""

        # Condition A: Check the Title
        for poison_word in JUNK_TITLES:
            # ✨ ADDED .lower() HERE JUST TO BE SAFE! ✨
            if poison_word.lower() in title_lower:
                delete_this_section = True
                reason = f"Title matched '{poison_word}'"
                break

        # Condition B: Check the Content (if Title was okay)
        if not delete_this_section:
            for poison_word in JUNK_CONTENT:
                # ✨ ADDED .lower() HERE JUST TO BE SAFE! ✨
                if poison_word.lower() in content_string:
                    delete_this_section = True
                    reason = f"Content matched '{poison_word}'"
                    break

        # Mark for deletion
        if delete_this_section:
            keys_to_delete.append((section_title, reason))

    # Actually delete them from the dictionary
    if keys_to_delete:
        print(f"🏥 {injury_name}:")
        for key, reason in keys_to_delete:
            del injury["sections"][key]
            total_deleted += 1
            print(f"  ❌ Deleted: '{key}' ({reason})")

# ==========================================
# 4. SAVE THE CLEANED DATABASE
# ==========================================
with open('pruned_master_database.json', 'w', encoding='utf-8') as f:
    json.dump(master_database, f, indent=4, ensure_ascii=False)

print("\n" + "="*50)
print(f"🎉 PRUNING COMPLETE! Destroyed {total_deleted} useless sections.")
print("💾 Saved to 'pruned_master_database.json'")
print("="*50)

✅ Loaded 16 injuries for targeted pruning.

🧹 Hunting for Ads and Footer Noise...

🏥 Hamstring Strain:
  ❌ Deleted: 'Introduction_Introductory_Text' (Title matched 'introduction_introductory_text')
🏥 Groin Strain:
  ❌ Deleted: 'Introduction_Introductory_Text' (Title matched 'introduction_introductory_text')
  ❌ Deleted: 'Exploring Shoulder Anatomy' (Content matched 'professional development from physiopedia plus')
  ❌ Deleted: 'Male Pelvic Anatomy' (Content matched 'get your ceus / cpd points from physiopedia')
  ❌ Deleted: 'Exploring Head and Jaw Anatomy' (Content matched 'ai supported learning powered by physiopedia')
  ❌ Deleted: '700+ accredited online courses for clinicians' (Title matched '700+')
  ❌ Deleted: 'Resources' (Title matched 'resources')
🏥 Calf Strain:
  ❌ Deleted: 'Introduction_Introductory_Text' (Title matched 'introduction_introductory_text')
  ❌ Deleted: 'The Theory and Practice of Massage and Exercise for Plantar Heel Pain' (Title matched 'pai')
  ❌ Deleted: 'Expl

In [ ]:
import json

# ==========================================
# 1. LOAD YOUR DATABASE
# ==========================================
try:
    with open('/content/pruned_master_database.json', 'r', encoding='utf-8') as f:
        master_database = json.load(f)
    print(f"✅ Loaded database.\n")
except FileNotFoundError:
    print(f"❌ Could not find 'pruned_master_database.json'.")
    master_database = []

# ==========================================
# 2. CHOP THE FINAL CONTENT BLOCK (SAFELY)
# ==========================================
print("✂️ Safely checking the final content block of each injury...\n")

total_blocks_chopped = 0

for injury in master_database:
    injury_name = injury.get("injury_name", "Unknown")
    sections = injury.get("sections", {})

    if not sections:
        continue

    # 1. Get the very last section title
    last_title = list(sections.keys())[-1]
    last_section_content = sections[last_title]

    if isinstance(last_section_content, list) and len(last_section_content) > 0:

        # 2. Peek at the very last block BEFORE deleting it
        last_block = last_section_content[-1]

        # Extract the text safely to check the first line
        block_text = ""
        if 'text' in last_block:
            block_text = last_block['text']
        elif 'list' in last_block and len(last_block['list']) > 0:
            block_text = str(last_block['list'][0])

        # Convert to lowercase to make the check bulletproof
        block_text_lower = block_text.strip().lower()

        # 3. THE CONDITIONAL CHECK
        # Only delete if it starts with "related articles" or "references"
        if block_text_lower.startswith("related articles") or block_text_lower.startswith("references"):

            # .pop() removes the final item safely
            removed_block = last_section_content.pop()
            total_blocks_chopped += 1

            print(f"🏥 {injury_name} -> Section '{last_title}':")
            print(f"  ❌ CHOPPED: Block matched junk words -> {str(removed_block)[:100]}...\n")

        else:
            print(f"🏥 {injury_name} -> Section '{last_title}':")
            print(f"  ✅ KEPT: Block is safe data -> '{block_text[:100]}...'\n")

    else:
        print(f"🏥 {injury_name} -> '{last_title}' was empty or not a list.\n")

# ==========================================
# 3. SAVE THE UPDATED DATABASE
# ==========================================
with open('final_block_chopped_database.json', 'w', encoding='utf-8') as f:
    json.dump(master_database, f, indent=4, ensure_ascii=False)

print("="*50)
print(f"🎉 SUCCESS! Safely chopped {total_blocks_chopped} junk blocks.")
print("💾 Saved to 'final_block_chopped_database.json'")
print("="*50)

✅ Loaded database.

✂️ Safely checking the final content block of each injury...

🏥 Hamstring Strain -> Section 'Criteria For Return to Play':
  ✅ KEPT: Block is safe data -> 'Clinicians should consider the history of hamstring injury before allowing return to play as previou...'

🏥 Groin Strain -> Section 'Conclusion':
  ✅ KEPT: Block is safe data -> 'Groin strains are common in sports, especially adductor muscle strain. Diagnosis should use the clin...'

🏥 Calf Strain -> Section 'Clinical Bottom Line':
  ❌ CHOPPED: Block matched junk words -> {'text': 'Related articles Triceps Surae - Physiopedia Introduction The triceps surae, a term used t...

🏥 Hip Flexor Strain -> Section 'Prognosis':
  ❌ CHOPPED: Block matched junk words -> {'text': 'Related articles Iliopsoas - Physiopedia Introduction A compound muscle composed of the il...

🏥 Quadriceps Contusion -> Section 'Clinical Bottom Line':
  ❌ CHOPPED: Block matched junk words -> {'text': 'Related articles Quadriceps Muscle Strain - P

# display after apply basic text cleansing and remove unwanted sector

In [ ]:
from IPython.display import display, HTML
import json
from datetime import datetime

# ==========================================
# ENHANCED DISPLAY FOR DYNAMIC DATABASE
# ==========================================
def display_dynamic_content(data):
    """Display dynamic scraped content in a beautiful format"""

    # Color scheme
    colors = {
        'primary': '#2c3e50',
        'secondary': '#34495e',
        'accent': '#3498db',
        'success': '#27ae60',
        'warning': '#f39c12',
        'danger': '#e74c3c',
        'light': '#ecf0f1',
        'dark': '#2c3e50',
        'info': '#00BCD4',
        'purple': '#9b59b6'
    }

    html = f"""
    <style>
        .dynamic-container {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1400px;
            margin: 20px auto;
            background: white;
            border-radius: 15px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
            overflow: hidden;
        }}
        .injury-header {{
            background: linear-gradient(135deg, {colors['primary']} 0%, {colors['secondary']} 100%);
            color: white;
            padding: 25px;
        }}
        .injury-title {{
            font-size: 2.2em;
            font-weight: 600;
            margin-bottom: 10px;
        }}
        .injury-meta {{
            display: flex;
            gap: 20px;
            font-size: 0.9em;
            opacity: 0.9;
        }}
        .stats-banner {{
            background: {colors['light']};
            padding: 15px 25px;
            display: flex;
            gap: 30px;
            border-bottom: 1px solid #ddd;
        }}
        .stat-item {{
            text-align: center;
        }}
        .stat-number {{
            font-size: 1.8em;
            font-weight: bold;
            color: {colors['primary']};
        }}
        .stat-label {{
            font-size: 0.8em;
            color: #666;
            text-transform: uppercase;
        }}
        .sections-container {{
            padding: 20px;
        }}
        .section-card {{
            background: white;
            border: 1px solid #e0e0e0;
            border-radius: 10px;
            margin-bottom: 20px;
            overflow: hidden;
            box-shadow: 0 2px 5px rgba(0,0,0,0.05);
        }}
        .section-header {{
            background: {colors['light']};
            padding: 15px 20px;
            border-bottom: 2px solid {colors['accent']};
            cursor: pointer;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}
        .section-header:hover {{
            background: #e9ecef;
        }}
        .section-title {{
            font-size: 1.2em;
            font-weight: 600;
            color: {colors['primary']};
        }}
        .section-badge {{
            background: {colors['accent']};
            color: white;
            padding: 3px 10px;
            border-radius: 15px;
            font-size: 0.8em;
        }}
        .section-content {{
            padding: 20px;
            display: block;
        }}
        .content-block {{
            margin-bottom: 15px;
            padding: 10px;
            border-left: 3px solid {colors['accent']};
            background: #f8f9fa;
            border-radius: 0 5px 5px 0;
        }}
        .text-block {{
            line-height: 1.6;
            color: #333;
        }}
        .list-block {{
            background: white;
            padding: 10px 20px;
            border-radius: 5px;
        }}
        .list-item {{
            padding: 5px 0;
            border-bottom: 1px dashed #eee;
        }}
        .list-item:last-child {{
            border-bottom: none;
        }}
        .table-block {{
            overflow-x: auto;
        }}
        .data-table {{
            width: 100%;
            border-collapse: collapse;
            background: white;
        }}
        .data-table th {{
            background: {colors['primary']};
            color: white;
            padding: 10px;
            text-align: left;
        }}
        .data-table td {{
            padding: 8px 10px;
            border: 1px solid #ddd;
        }}
        .data-table tr:nth-child(even) {{
            background: #f8f9fa;
        }}
        .content-counter {{
            display: inline-block;
            background: {colors['info']};
            color: white;
            padding: 2px 8px;
            border-radius: 12px;
            font-size: 0.7em;
            margin-left: 10px;
        }}
        .toggle-btn {{
            background: none;
            border: none;
            color: {colors['primary']};
            font-size: 1.2em;
            cursor: pointer;
        }}
        .summary-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin-top: 20px;
        }}
        .summary-card {{
            background: white;
            border-radius: 8px;
            padding: 15px;
            text-align: center;
            box-shadow: 0 2px 5px rgba(0,0,0,0.05);
        }}
        .section-type-badge {{
            display: inline-block;
            padding: 2px 8px;
            border-radius: 12px;
            font-size: 0.7em;
            font-weight: bold;
            margin-right: 5px;
        }}
        .type-text {{ background: {colors['accent']}; color: white; }}
        .type-list {{ background: {colors['success']}; color: white; }}
        .type-table {{ background: {colors['purple']}; color: white; }}
    </style>
    """

    for injury in data:
        injury_name = injury.get('injury_name', 'Unknown')
        source_url = injury.get('source_url', '')
        sections = injury.get('sections', {})

        # Calculate stats
        total_sections = len(sections)
        total_text_blocks = 0
        total_list_items = 0
        total_tables = 0

        for section_name, content in sections.items():
            for item in content:
                if 'text' in item:
                    total_text_blocks += 1
                elif 'list' in item:
                    total_list_items += len(item['list'])
                elif 'table' in item:
                    total_tables += 1

        html += f"""
        <div class="dynamic-container">
            <div class="injury-header">
                <div class="injury-title">🏥 {injury_name}</div>
                <div class="injury-meta">
                    <span>🔗 <a href="{source_url}" style="color: white;" target="_blank">{source_url}</a></span>
                    <span>📅 Scraped: {datetime.now().strftime('%Y-%m-%d %H:%M')}</span>
                </div>
            </div>

            <div class="stats-banner">
                <div class="stat-item">
                    <div class="stat-number">{total_sections}</div>
                    <div class="stat-label">Sections</div>
                </div>
                <div class="stat-item">
                    <div class="stat-number">{total_text_blocks}</div>
                    <div class="stat-label">Text Blocks</div>
                </div>
                <div class="stat-item">
                    <div class="stat-number">{total_list_items}</div>
                    <div class="stat-label">List Items</div>
                </div>
                <div class="stat-item">
                    <div class="stat-number">{total_tables}</div>
                    <div class="stat-label">Tables</div>
                </div>
            </div>

            <div class="sections-container">
        """

        # Display each section
        for section_name, content in sections.items():
            content_types = []
            if any('text' in item for item in content):
                content_types.append('📝 Text')
            if any('list' in item for item in content):
                content_types.append('📋 List')
            if any('table' in item for item in content):
                content_types.append('📊 Table')

            html += f"""
                <div class="section-card">
                    <div class="section-header" onclick="this.nextElementSibling.style.display = this.nextElementSibling.style.display == 'none' ? 'block' : 'none'">
                        <div>
                            <span class="section-title">{section_name}</span>
                            <span class="content-counter">{len(content)} items</span>
                        </div>
                        <div>
                            <span class="section-badge">{' · '.join(content_types)}</span>
                            <span class="toggle-btn">▼</span>
                        </div>
                    </div>
                    <div class="section-content">
            """

            # Display content items
            for item in content:
                if 'text' in item:
                    html += f"""
                        <div class="content-block">
                            <span class="section-type-badge type-text">TEXT</span>
                            <div class="text-block">{item['text']}</div>
                        </div>
                    """

                elif 'list' in item:
                    html += f"""
                        <div class="content-block">
                            <span class="section-type-badge type-list">LIST</span>
                            <div class="list-block">
                    """
                    for li in item['list']:
                        html += f'<div class="list-item">• {li}</div>'
                    html += "</div></div>"

                elif 'table' in item:
                    table = item['table']
                    html += f"""
                        <div class="content-block">
                            <span class="section-type-badge type-table">TABLE</span>
                            <div class="table-block">
                                <table class="data-table">
                    """
                    if table['header']:
                        html += "<tr>" + "".join([f"<th>{h}</th>" for h in table['header']]) + "</tr>"
                    for row in table['rows']:
                        html += "<tr>" + "".join([f"<td>{cell}</td>" for cell in row]) + "</tr>"
                    html += "</table></div></div>"

            html += "</div></div>"

        # Add summary section
        html += f"""
                <div class="summary-grid">
                    <div class="summary-card">
                        <h4 style="color: {colors['primary']};">📊 Content Distribution</h4>
                        <div style="margin-top: 10px;">
                            <div>📝 Text: {total_text_blocks}</div>
                            <div>📋 List Items: {total_list_items}</div>
                            <div>📊 Tables: {total_tables}</div>
                        </div>
                    </div>
                    <div class="summary-card">
                        <h4 style="color: {colors['primary']};">📈 Section Types</h4>
                        <div style="margin-top: 10px;">
                            {len([s for s in sections if 'Introduction' in s])} Introduction Sections<br>
                            {len([s for s in sections if any(x in s.lower() for x in ['management', 'treatment'])])} Treatment Sections<br>
                            {len([s for s in sections if 'diagnosis' in s.lower()])} Diagnosis Sections
                        </div>
                    </div>
                </div>
            </div>
        </div>
        """

    return HTML(html)

def display_dynamic_summary(data):
    """Display a quick summary of all injuries"""
    print("\n" + "🔥"*60)
    print("🔥 DYNAMIC DATABASE SUMMARY")
    print("🔥"*60)

    for i, injury in enumerate(data, 1):
        injury_name = injury.get('injury_name', 'Unknown')
        sections = injury.get('sections', {})

        # Count content types
        text_count = 0
        list_count = 0
        table_count = 0

        for section_content in sections.values():
            for item in section_content:
                if 'text' in item:
                    text_count += 1
                elif 'list' in item:
                    list_count += len(item['list'])
                elif 'table' in item:
                    table_count += 1

        print(f"\n{i}. 🏥 {injury_name}")
        print(f"   📚 Sections: {len(sections)}")
        print(f"   📝 Text blocks: {text_count}")
        print(f"   📋 List items: {list_count}")
        print(f"   📊 Tables: {table_count}")
        print(f"   🔗 URL: {injury.get('source_url', 'N/A')}")

        # Show first 3 sections as preview
        section_names = list(sections.keys())[:3]
        if section_names:
            print(f"   📑 First sections: {', '.join(section_names)}")
        if len(sections) > 3:
            print(f"   ... and {len(sections)-3} more sections")

    print("\n" + "🔥"*60)

def display_sample_content(data, injury_index=0, sample_sections=3):
    """Display a sample of content from a specific injury"""
    if not data or injury_index >= len(data):
        print("❌ Invalid injury index")
        return

    injury = data[injury_index]
    injury_name = injury.get('injury_name', 'Unknown')
    sections = injury.get('sections', {})

    print("\n" + "="*60)
    print(f"📋 SAMPLE CONTENT: {injury_name}")
    print("="*60)

    section_items = list(sections.items())[:sample_sections]
    for section_name, content in section_items:
        print(f"\n📌 {section_name}")
        print("-" * 40)

        for i, item in enumerate(content[:2]):  # Show first 2 items per section
            if 'text' in item:
                print(f"   📝 {item['text'][:150]}...")
            elif 'list' in item:
                print(f"   📋 List with {len(item['list'])} items:")
                for li in item['list'][:3]:  # Show first 3 list items
                    print(f"      • {li[:50]}...")
            elif 'table' in item:
                table = item['table']
                print(f"   📊 Table with {len(table['rows'])} rows")

        if len(content) > 2:
            print(f"   ... and {len(content)-2} more items")

    print("\n" + "="*60)

# ==========================================
# MAIN DISPLAY EXECUTION
# ==========================================
print("\n" + ""*60)
print(" DISPLAYING DYNAMIC SCRAPED DATABASE")
print(""*60)

# Load the data
try:
    with open('final_block_chopped_database.json', 'r', encoding='utf-8') as f:
        dynamic_data = json.load(f)
    print(f"✅ Loaded {len(dynamic_data)} injuries from database\n")
except FileNotFoundError:
    print("❌ Database file not found. Using the scraped data from memory.")
    dynamic_data = master_dynamic_database

# Option 1: Beautiful HTML display (full)
print("\n📊 OPTION 1: FULL HTML DISPLAY")
print("-" * 40)
display(display_dynamic_content(dynamic_data))

# Option 2: Quick summary
print("\n📊 OPTION 2: DATABASE SUMMARY")
print("-" * 40)
display_dynamic_summary(dynamic_data)

# Option 3: Sample content from first injury
print("\n📊 OPTION 3: SAMPLE CONTENT (First Injury)")
print("-" * 40)
display_sample_content(dynamic_data, injury_index=0, sample_sections=3)

# Option 4: Raw JSON structure preview
print("\n📊 OPTION 4: RAW JSON PREVIEW")
print("-" * 40)
print(json.dumps(dynamic_data[0] if dynamic_data else {}, indent=2)[:1000] + "...")

print("\n✅ Display complete!")


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🔥 DISPLAYING DYNAMIC SCRAPED DATABASE
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
✅ Loaded 16 injuries from database


📊 OPTION 1: FULL HTML DISPLAY
----------------------------------------



📊 OPTION 2: DATABASE SUMMARY
----------------------------------------

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🔥 DYNAMIC DATABASE SUMMARY
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

1. 🏥 Hamstring Strain
   📚 Sections: 16
   📝 Text blocks: 58
   📋 List items: 111
   📊 Tables: 2
   🔗 URL: https://www.physio-pedia.com/Hamstring_Strain
   📑 First sections: Introduction, Epidemiology/Etiology, Predisposing Factors/Risk Factors
   ... and 13 more sections

2. 🏥 Groin Strain
   📚 Sections: 17
   📝 Text blocks: 51
   📋 List items: 18
   📊 Tables: 5
   🔗 URL: https://www.physio-pedia.com/Groin_Strain
   📑 First sections: Definition/Description, Clinically Relevant Anatomy, Epidemiology /Etiology
   ... and 14 more sections

3. 🏥 Calf Strain
   📚 Sections: 14
   📝 Text blocks: 36
   📋 List items: 46
   📊 Tables: 1
   🔗 URL: https://www.physio-pedia.com/Calf_Strain
   📑 First sections: Description, Clinically Relevant Anatomy, Epidemiology/Etiology
   ... 

# all parent name in each exercise that will merged into 10 standard sector

In [ ]:
import json

# ==========================================
# 1. LOAD THE CHOPPED DATABASE
# ==========================================
try:
    with open('final_block_chopped_database.json', 'r', encoding='utf-8') as f:
        chopped_database = json.load(f)
    print(f"✅ Loaded 'final_block_chopped_database.json'\n")
except FileNotFoundError:
    print(f"❌ Could not find 'final_block_chopped_database.json'.")
    chopped_database = []

# ==========================================
# 2. DISPLAY SUMMARY OF ALL INJURIES AND SECTIONS
# ==========================================
print("="*60)
print("📊 FINAL DATABASE SUMMARY")
print("="*60)

total_injuries = len(chopped_database)
total_sections = 0
total_blocks = 0

for idx, injury in enumerate(chopped_database, 1):
    injury_name = injury.get("injury_name", "Unknown")
    sections = injury.get("sections", {})

    # Count sections and blocks
    num_sections = len(sections)
    total_sections += num_sections

    section_blocks = []
    for section_title, content in sections.items():
        if isinstance(content, list):
            block_count = len(content)
            total_blocks += block_count
            section_blocks.append(f"  • {section_title}: {block_count} blocks")

    # Print injury summary
    print(f"\n🏥 {idx}. {injury_name}")
    print(f"   Sections: {num_sections}")
    if section_blocks:
        for section in section_blocks:
            print(f"   {section}")
    else:
        print("   No sections with content")

# ==========================================
# 3. FINAL STATISTICS
# ==========================================
print("\n" + "="*60)
print("📈 STATISTICS")
print("="*60)
print(f"Total Injuries: {total_injuries}")
print(f"Total Sections: {total_sections}")
print(f"Total Content Blocks: {total_blocks}")
print("="*60)

# Optional: Show sample of first injury's first section
if chopped_database and chopped_database[0].get("sections"):
    first_injury = chopped_database[0]
    first_section = list(first_injury["sections"].keys())[0] if first_injury["sections"] else None

    if first_section:
        print(f"\n📝 PREVIEW: '{first_injury.get('injury_name')}' - '{first_section}'")
        first_content = first_injury["sections"][first_section]
        if first_content and len(first_content) > 0:
            print(f"   First block: {str(first_content[0])[:150]}...")

✅ Loaded 'final_block_chopped_database.json'

📊 FINAL DATABASE SUMMARY

🏥 1. Hamstring Strain
   Sections: 16
     • Introduction: 6 blocks
     • Epidemiology/Etiology: 1 blocks
     • Predisposing Factors/Risk Factors: 5 blocks
     • Characteristics/Clinical Presentation: 5 blocks
     • Differential Diagnosis: 5 blocks
     • Diagnostic Procedures: 6 blocks
     • Outcome Measures: 1 blocks
     • Examination: 13 blocks
     • Medical Management: 2 blocks
     • Prognosis: 3 blocks
     • Physiotherapy Management: 10 blocks
     • Phase I (week 0-3): 8 blocks
     • Phase 2 (week 3-12): 11 blocks
     • Phase 3 (week 12+): 10 blocks
     • Prevention Of Hamstring Injuries: 1 blocks
     • Criteria For Return to Play: 1 blocks

🏥 2. Groin Strain
   Sections: 17
     • Definition/Description: 6 blocks
     • Clinically Relevant Anatomy: 7 blocks
     • Epidemiology /Etiology: 4 blocks
     • Characteristics/Clinical Presentation: 6 blocks
     • Differential Diagnosis: 5 blocks
     

In [ ]:
import json

# ==========================================
# 1. LOAD THE DATABASE
# ==========================================
try:
    with open('final_block_chopped_database.json', 'r', encoding='utf-8') as f:
        database = json.load(f)
    print(f"✅ Loaded 'final_block_chopped_database.json'\n")
except FileNotFoundError:
    print(f"❌ Could not find 'final_block_chopped_database.json'.")
    database = []

# ==========================================
# 2. COLLECT ALL UNIQUE SECTION NAMES
# ==========================================
unique_sections = set()

for injury in database:
    sections = injury.get("sections", {})
    for section_name in sections.keys():
        unique_sections.add(section_name)

# ==========================================
# 3. DISPLAY UNIQUE SECTION NAMES
# ==========================================
print("="*50)
print("📋 UNIQUE SECTION NAMES")
print("="*50)

# Sort alphabetically for better display
sorted_sections = sorted(unique_sections)

for idx, section in enumerate(sorted_sections, 1):
    print(f"{idx:3d}. {section}")

print("\n" + "="*50)
print(f"Total Unique Sections: {len(unique_sections)}")
print("="*50)

# Optional: Show which injuries have each section
print("\n🔍 Section Distribution:")
print("-"*50)

for section in sorted_sections[:5]:  # Show first 5 sections as example
    injuries_with_section = []
    for injury in database:
        if section in injury.get("sections", {}):
            injuries_with_section.append(injury.get("injury_name", "Unknown"))

    print(f"\n📌 '{section}'")
    print(f"   Found in {len(injuries_with_section)} injuries:")
    for injury in injuries_with_section[:3]:  # Show first 3 injuries as example
        print(f"   • {injury}")
    if len(injuries_with_section) > 3:
        print(f"   • ... and {len(injuries_with_section) - 3} more")

✅ Loaded 'final_block_chopped_database.json'

📋 UNIQUE SECTION NAMES
  1. 1. Radiographs
  2. 2. MRI
  3. 3. Instrumented laxity testing/arthrometric evaluation of the knee
  4. 4. Dynamic Ultrasonography
  5. Acute Management
  6. Acute Phase
  7. Adductor-related groin strain injury program
  8. Adjunct Therapies
  9. Aetiology
 10. Ankle Bracing and Taping
 11. Any previous injury
 12. Assess the Effectiveness of Intervention
 13. Assessment
 14. Associated Injuries
 15. Balance & Agility
 16. Biomechanics of Injury
 17. Bone Contusions and Microfractures
 18. Case Studies
 19. Characteristics/ clinical presentation
 20. Characteristics/Clinical Presentation
 21. Chondral Injury
 22. Chronic Ankle Instability
 23. Classification
 24. Classification of Ankle Sprain
 25. Classification of Syndesmosis Sprains
 26. Clinical Assessment Tools to Identify At‐Risk Athletes
 27. Clinical Bottom Line
 28. Clinical Examination
 29. Clinical Presentation
 30. Clinical Relevant Anatomy
 31. Clin

# merge messy sector into 10

In [ ]:
import json

# ==========================================
# 1. LOAD YOUR PRUNED DATABASE
# ==========================================
try:
    with open('final_block_chopped_database.json', 'r', encoding='utf-8') as f:
        master_database = json.load(f)
    print("✅ Loaded database for Semantic Bucketing.\n")
except FileNotFoundError:
    print("❌ Could not find database. Make sure the filename is correct.")
    master_database = []

# ==========================================
# 2. THE TRASH CAN & TAXONOMY MAP
# ==========================================
# Any title containing these words will be permanently DELETED
SKIP_LIST = [
    "course page funded",
    "plus online learning",
    "case studies",
    "key research"
]

TAXONOMY = {
    "1_Introduction_Definition": ["introduction", "description", "definition", "background", "what is", "terminology", "location"],
    "2_Anatomy": ["anatomy", "locoregional pathologies", "functions of", "adductors", "abductors"],
    "3_Mechanism_Etiology": ["etiology", "epidemiology", "mechanism", "pathological", "pathophysiology", "aetiology", "biomechanics", "causes", "classification", "grade", "grading", "fractures", "contusions", "lesions", "injury", "rupture", "strain", "tear", "syndrome", "instability", "tendinopathy", "cyst", "associated injuries"],
    "4_Risk_Factors": ["risk", "predisposing", "imbalance"],
    "5_Clinical_Presentation": ["presentation", "characteristics", "symptoms", "signs"],
    "6_Examination_Diagnosis": ["examination", "diagnostic", "imaging", "assessment", "tests", "testing", "evaluation", "screening", "mri", "radiographs", "analysis", "outcome measures", "squeeze test", "ultrasonography", "laboratory"],
    "7_Differential_Diagnosis": ["differential"],
    "8_Medical_Management": ["medical management", "surgical", "pharmacological", "medication", "injection", "operative", "conservative", "drugs", "corticosteroid", "surgery", "lidocaine", "platelet", "sclerotherapy", "invasive", "other"], # Added "other" here
    "9_Rehabilitation_Physiotherapy": ["physiotherapy", "rehabilitation", "phase", "protocol", "exercises", "management", "therapy", "training", "program", "splints", "taping", "mobilizations", "strength", "flexibility", "balance", "agility", "principles", "fifa", "pep", "sportsmetric", "harmoknee", "bracing", "loading", "adjunct", "education", "lifts", "iontophoresis", "needling", "part "],
    "10_Prevention_Return_To_Play": ["prevention", "return to", "rtp", "rts", "prognosis", "consequences", "conclusion", "recommendations", "effectiveness", "bottom line", "evidence", "take-home", "considerations"] # Added "take-home" and "considerations" here
}

# ==========================================
# 3. THE MERGING ENGINE
# ==========================================
standardized_database = []
total_uncategorized = 0
total_skipped = 0

print("🗄️ Sorting titles into 10 Master Buckets (and taking out the trash)...\n")

for injury in master_database:
    injury_name = injury.get("injury_name", "Unknown")
    old_sections = injury.get("sections", {})

    new_sections = {bucket: [] for bucket in TAXONOMY.keys()}
    new_sections["Uncategorized"] = []

    for old_title, blocks in old_sections.items():
        title_lower = old_title.lower()

        # 🛑 Check the Trash Can first
        should_skip = False
        for junk_word in SKIP_LIST:
            if junk_word in title_lower:
                should_skip = True
                total_skipped += 1
                print(f"  🗑️ Trashed ad/junk section: '{old_title}'")
                break

        if should_skip:
            continue # Skip processing this section entirely!

        # 🗄️ Sort into Buckets
        matched_bucket = "Uncategorized"
        for master_bucket, keywords in TAXONOMY.items():
            if any(keyword in title_lower for keyword in keywords):
                matched_bucket = master_bucket
                break

        if matched_bucket == "Uncategorized":
            total_uncategorized += 1
            print(f"  ⚠️ Warning: '{old_title}' didn't match any keywords. Sent to Uncategorized.")

        if blocks and isinstance(blocks, list):
            blocks.insert(0, {"text": f"\n=== {old_title.upper()} ==="})

        new_sections[matched_bucket].extend(blocks)

    final_sections = {k: v for k, v in new_sections.items() if len(v) > 0}
    sorted_final_sections = dict(sorted(final_sections.items()))

    standardized_database.append({
        "injury_name": injury_name,
        "source_url": injury.get("source_url", ""),
        "sections": sorted_final_sections
    })

# ==========================================
# 4. SAVE THE FINAL STRUCTURE
# ==========================================
with open('standardized_master_database.json', 'w', encoding='utf-8') as f:
    json.dump(standardized_database, f, indent=4, ensure_ascii=False)

print("\n" + "="*50)
print(f"🎉 STANDARDIZATION COMPLETE!")
print(f"🗑️ Junk sections deleted: {total_skipped}")
if total_uncategorized == 0:
    print("✨ Flawless victory! 0 Uncategorized sections.")
print("💾 Saved to 'standardized_master_database.json'")
print("="*50)

✅ Loaded database for Semantic Bucketing.

🗄️ Sorting titles into 10 Master Buckets (and taking out the trash)...

  🗑️ Trashed ad/junk section: 'Key Research'
  🗑️ Trashed ad/junk section: 'Case Studies'
  🗑️ Trashed ad/junk section: 'This is a course page funded by Plus online learning'

🎉 STANDARDIZATION COMPLETE!
🗑️ Junk sections deleted: 3
✨ Flawless victory! 0 Uncategorized sections.
💾 Saved to 'standardized_master_database.json'


# disply the tree architecture of json data after merge and clean in standard form

In [ ]:
import json
import re
from rich.console import Console
from rich.tree import Tree
from rich.text import Text
from rich.panel import Panel
from rich.style import Style

console = Console()

def natural_sort_key(item):
    section_name = item[0]
    # Look for a number at the start of the section name
    match = re.search(r'^(\d+)', section_name)
    if match:
        return int(match.group(1))
    return 999  # If no number, push to the bottom

# Load your generated JSON file
try:
    with open('data.sjon', 'r', encoding='utf-8') as f:
        master_data = json.load(f)
except FileNotFoundError:
    console.print("[bold red]Error: ' not found.[/bold red]")
    exit()

# Create the root of the tree - using bright white for better visibility
root = Tree("[black]PHYSIO-PEDIA INJURY DATABASE[/black]", guide_style="white")

# Loop through each parent (injury) and its children (sections)
for injury_index, injury in enumerate(master_data, 1):
    injury_name = injury.get("injury_name", "Unknown Injury")

    # Add the Parent node with numbering - using bright white for maximum visibility
    injury_node = root.add(f"[black]{injury_index}. {injury_name}[/black]")

    # Add the Child nodes (sections)
    sections = injury.get("sections", {})
    if not sections:
        injury_node.add("[dim]No content available[/dim]")
        continue

    # Sort sections using our natural sort key
    sorted_sections = sorted(sections.items(), key=natural_sort_key)

    for section_index, (section_name, content_list) in enumerate(sorted_sections, 1):
        item_count = len(content_list)

        # Create section display with high contrast colors
        if item_count > 0:
            section_text = Text()
            section_text.append(f"{section_index}. {section_name} ", style="cyan")  # Cyan for good visibility
            section_text.append(f"[{item_count}]", style="yellow")  # Yellow for counts
            injury_node.add(section_text)
        else:
            section_text = Text(f"{section_index}. {section_name} ", style="dim white")
            section_text.append("[empty]", style="red")
            injury_node.add(section_text)

# Print the enhanced tree to the console
console.print("\n")
console.print(root)
console.print("\n")

# Add summary statistics at the bottom
total_injuries = len(master_data)
total_sections = sum(len(injury.get("sections", {})) for injury in master_data)
total_blocks = sum(
    len(content)
    for injury in master_data
    for content in injury.get("sections", {}).values()
)

summary_panel = Panel(
    f"[black]Total Injuries:[/black] {total_injuries}   |   "
    f"[black]Total Sections:[/black] {total_sections}   |   "
    f"[black]Total Content Blocks:[/black] {total_blocks}",
    title="Database Summary",
    border_style="white",
    expand=False
)
console.print(summary_panel)

PHYSIO-PEDIA INJURY DATABASE
├── 1. Hamstring Strain
│   ├── 1. 1_Introduction_Definition [7]
│   ├── 2. 3_Mechanism_Etiology [2]
│   ├── 3. 4_Risk_Factors [6]
│   ├── 4. 5_Clinical_Presentation [6]
│   ├── 5. 6_Examination_Diagnosis [23]
│   ├── 6. 7_Differential_Diagnosis [6]
│   ├── 7. 8_Medical_Management [14]
│   ├── 8. 9_Rehabilitation_Physiotherapy [32]
│   └── 9. 10_Prevention_Return_To_Play [8]
├── 2. Groin Strain
│   ├── 1. 1_Introduction_Definition [7]
│   ├── 2. 2_Anatomy [18]
│   ├── 3. 3_Mechanism_Etiology [18]
│   ├── 4. 5_Clinical_Presentation [7]
│   ├── 5. 6_Examination_Diagnosis [9]
│   ├── 6. 7_Differential_Diagnosis [6]
│   ├── 7. 8_Medical_Management [8]
│   ├── 8. 9_Rehabilitation_Physiotherapy [2]
│   └── 9. 10_Prevention_Return_To_Play [11]
├── 3. Calf Strain
│   ├── 1. 1_Introduction_Definition [4]
│   ├── 2. 2_Anatomy [4]
│   ├── 3. 3_Mechanism_Etiology [19]
│   ├── 4. 4_Risk_Factors [6]
│   ├── 5. 5_Clinical_Presentation [4]
│   ├── 6. 6_Examination_Diagnosis [2]
│   ├── 7. 7_Differential_Diagnosis [3]
│   ├── 8. 8_Medical_Management [5]
│   ├── 9. 9_Rehabilitation_Physiotherapy [12]
│   └── 10. 10_Prevention_Return_To_Play [3]
├── 4. Hip Flexor Strain
│   ├── 1. 1_Introduction_Definition [3]
│   ├── 2. 2_Anatomy [15]
│   ├── 3. 3_Mechanism_Etiology [11]
│   ├── 4. 5_Clinical_Presentation [2]
│   ├── 5. 6_Examination_Diagnosis [15]
│   ├── 6. 7_Differential_Diagnosis [4]
│   ├── 7. 8_Medical_Management [8]
│   ├── 8. 9_Rehabilitation_Physiotherapy [6]
│   └── 9. 10_Prevention_Return_To_Play [4]
├── 5. Quadriceps Contusion
│   ├── 1. 1_Introduction_Definition [4]
│   ├── 2. 3_Mechanism_Etiology [12]
│   ├── 3. 4_Risk_Factors [2]
│   ├── 4. 5_Clinical_Presentation [4]
│   ├── 5. 6_Examination_Diagnosis [6]
│   ├── 6. 9_Rehabilitation_Physiotherapy [6]
│   └── 7. 10_Prevention_Return_To_Play [4]
├── 6. Metatarsal Fracture
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 2_Anatomy [2]
│   ├── 3. 3_Mechanism_Etiology [10]
│   ├── 4. 5_Clinical_Presentation [4]
│   ├── 5. 6_Examination_Diagnosis [24]
│   ├── 6. 7_Differential_Diagnosis [9]
│   ├── 7. 8_Medical_Management [2]
│   └── 8. 9_Rehabilitation_Physiotherapy [6]
├── 7. Concussion
│   ├── 1. 1_Introduction_Definition [4]
│   ├── 2. 3_Mechanism_Etiology [5]
│   ├── 3. 4_Risk_Factors [6]
│   ├── 4. 5_Clinical_Presentation [12]
│   ├── 5. 6_Examination_Diagnosis [55]
│   ├── 6. 8_Medical_Management [2]
│   ├── 7. 9_Rehabilitation_Physiotherapy [22]
│   └── 8. 10_Prevention_Return_To_Play [6]
├── 8. ACL Injury
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 2_Anatomy [9]
│   ├── 3. 3_Mechanism_Etiology [75]
│   ├── 4. 4_Risk_Factors [20]
│   ├── 5. 5_Clinical_Presentation [2]
│   ├── 6. 6_Examination_Diagnosis [42]
│   ├── 7. 7_Differential_Diagnosis [7]
│   ├── 8. 9_Rehabilitation_Physiotherapy [25]
│   └── 9. 10_Prevention_Return_To_Play [10]
├── 9. Meniscus Lesion
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 2_Anatomy [5]
│   ├── 3. 3_Mechanism_Etiology [22]
│   ├── 4. 4_Risk_Factors [2]
│   ├── 5. 5_Clinical_Presentation [3]
│   ├── 6. 6_Examination_Diagnosis [29]
│   ├── 7. 7_Differential_Diagnosis [6]
│   ├── 8. 8_Medical_Management [22]
│   ├── 9. 9_Rehabilitation_Physiotherapy [26]
│   └── 10. 10_Prevention_Return_To_Play [5]
├── 10. MCL Sprain
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 3_Mechanism_Etiology [6]
│   ├── 3. 5_Clinical_Presentation [7]
│   ├── 4. 6_Examination_Diagnosis [14]
│   ├── 5. 7_Differential_Diagnosis [6]
│   ├── 6. 8_Medical_Management [7]
│   ├── 7. 9_Rehabilitation_Physiotherapy [19]
│   └── 8. 10_Prevention_Return_To_Play [4]
├── 11. Patellar Tendinopathy
│   ├── 1. 1_Introduction_Definition [3]
│   ├── 2. 2_Anatomy [6]
│   ├── 3. 3_Mechanism_Etiology [9]
│   ├── 4. 5_Clinical_Presentation [5]
│   ├── 5. 6_Examination_Diagnosis [8]
│   ├── 6. 7_Differential_Diagnosis [2]
│   ├── 7. 8_Medical_Management [11]
│   ├── 8. 9_Rehabilitation_Physiotherapy [15]
│   └── 9. 10_Prevention_Return_To_Play [2

╭────────────────────────────── Database Summary ───────────────────────────────╮
│ Total Injuries: 16   |   Total Sections: 137   |   Total Content Blocks: 1350 │
╰───────────────────────────────────────────────────────────────────────────────╯

# save data into drive

In [ ]:
import shutil
import os

print("\n" + "="*50)
print("COPYING FILES TO GOOGLE DRIVE")
print("="*50)

# Files to copy
files_to_copy = [
    "standardized_master_database.json"
]

destination = '/content/drive/MyDrive/SPORT_METADATA/Structure data/Rehab&Recovery'


# Copy each file
for file_path in files_to_copy:
    file_name = os.path.basename(file_path)
    print(f"\n📄 Copying: {file_name}...", end=" ")

    if os.path.exists(file_path):
        shutil.copy2(file_path, destination)
        print("✅ Done")
    else:
        print("❌ File not found")

print("\n" + "="*50)
print("✅ All files copied to:", destination)
print("="*50)


COPYING FILES TO GOOGLE DRIVE

📄 Copying: standardized_master_database.json... ✅ Done

✅ All files copied to: /content/drive/MyDrive/SPORT_METADATA/Structure data/Rehab&Recovery


In [ ]:
from IPython.display import display, HTML
import json
from datetime import datetime

def display_injury_by_source(data, target_url):
    """
    Display a specific injury using its source URL as a unique key,
    formatted specifically for the standardized 9-section clinical structure.
    """

    # 1. Find the specific injury by URL
    injury = next((item for item in data if item.get('source_url') == target_url), None)

    if not injury:
        return HTML(f"""
        <div style="padding: 20px; background: #fee2e2; border-left: 5px solid #ef4444; border-radius: 5px; font-family: sans-serif;">
            <h3 style="color: #991b1b; margin-top: 0;">❌ Injury Not Found</h3>
            <p style="color: #7f1d1d;">No record found with the source URL: <b>{target_url}</b></p>
        </div>
        """)

    injury_name = injury.get('injury_name', 'Unknown Injury')
    sections = injury.get('sections', {})

    # Define the strict ordering and icons for your standard sections
    section_config = {
        "1_Introduction_Definition": {"icon": "📖", "label": "Introduction & Definition"},
        "3_Mechanism_Etiology": {"icon": "⚙️", "label": "Mechanism & Etiology"},
        "4_Risk_Factors": {"icon": "⚠️", "label": "Risk Factors"},
        "5_Clinical_Presentation": {"icon": "🤒", "label": "Clinical Presentation"},
        "6_Examination_Diagnosis": {"icon": "🩺", "label": "Examination & Diagnosis"},
        "7_Differential_Diagnosis": {"icon": "🔍", "label": "Differential Diagnosis"},
        "8_Medical_Management": {"icon": "💊", "label": "Medical Management"},
        "9_Rehabilitation_Physiotherapy": {"icon": "🏃‍♂️", "label": "Rehabilitation & Physiotherapy"},
        "10_Prevention_Return_To_Play": {"icon": "🛡️", "label": "Prevention & Return to Play"}
    }

    # Color scheme
    colors = {
        'primary': '#1e3a8a', 'secondary': '#3b82f6', 'accent': '#0284c7',
        'light': '#f3f4f6', 'dark': '#1f2937', 'border': '#e5e7eb'
    }

    # Base HTML & CSS
    html = f"""
    <style>
        .med-container {{ font-family: 'Segoe UI', system-ui, sans-serif; max-width: 1200px; margin: 20px auto; background: white; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); overflow: hidden; color: {colors['dark']}; }}
        .med-header {{ background: linear-gradient(135deg, {colors['primary']}, {colors['secondary']}); color: white; padding: 30px; }}
        .med-title {{ font-size: 2.5em; font-weight: 700; margin: 0 0 10px 0; }}
        .med-meta {{ font-size: 0.9em; opacity: 0.9; display: flex; gap: 20px; }}
        .med-meta a {{ color: #bfdbfe; text-decoration: none; }}
        .med-meta a:hover {{ text-decoration: underline; }}

        /* Tree Index Styles */
        .tree-container {{ background: #f8fafc; padding: 25px 30px; border-bottom: 1px solid {colors['border']}; }}
        .tree-title {{ font-size: 1.2em; font-weight: 600; color: {colors['primary']}; margin-bottom: 15px; border-bottom: 2px solid {colors['accent']}; padding-bottom: 5px; display: inline-block; }}
        .tree-list {{ list-style: none; padding-left: 10px; margin: 0; font-family: monospace; font-size: 14px; line-height: 1.8; }}
        .tree-item {{ position: relative; padding-left: 20px; }}
        .tree-item::before {{ content: "├──"; position: absolute; left: -10px; color: #94a3b8; }}
        .tree-item:last-child::before {{ content: "└──"; }}
        .tree-count {{ color: #ef4444; font-weight: bold; }}
        .tree-link {{ text-decoration: none; color: {colors['dark']}; transition: color 0.2s; }}
        .tree-link:hover {{ color: {colors['secondary']}; font-weight: bold; }}

        /* Content Sections */
        .content-body {{ padding: 30px; }}
        .section-box {{ margin-bottom: 30px; border: 1px solid {colors['border']}; border-radius: 8px; overflow: hidden; }}
        .section-head {{ background: {colors['light']}; padding: 15px 20px; font-size: 1.3em; font-weight: 600; color: {colors['primary']}; border-bottom: 1px solid {colors['border']}; display: flex; justify-content: space-between; align-items: center; }}
        .section-content {{ padding: 20px; }}

        /* Block Types */
        .block-text {{ margin-bottom: 15px; line-height: 1.6; padding: 12px; background: #fff; border-left: 4px solid #3b82f6; border-radius: 0 6px 6px 0; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }}
        .block-list {{ margin-bottom: 15px; padding: 15px 20px; background: #f8fafc; border-radius: 6px; border: 1px solid #e2e8f0; }}
        .block-list ul {{ margin: 0; padding-left: 20px; }}
        .block-list li {{ margin-bottom: 5px; line-height: 1.5; }}
        .badge-count {{ background: #e0f2fe; color: #0369a1; font-size: 0.7em; padding: 4px 10px; border-radius: 12px; vertical-align: middle; }}
    </style>

    <div class="med-container">
        <div class="med-header">
            <h1 class="med-title">🏥 {injury_name}</h1>
            <div class="med-meta">
                <span>🔗 <a href="{target_url}" target="_blank">Source URL</a></span>
                <span>📅 Generated: {datetime.now().strftime('%Y-%m-%d')}</span>
            </div>
        </div>
    """

    # 2. Build the Tree Index
    html += """
        <div class="tree-container">
            <div class="tree-title">📑 Clinical Structure Overview</div>
            <ul class="tree-list">
    """

    # Generate tree based on config order
    for idx, (sec_key, config) in enumerate(section_config.items(), 1):
        # Find matching section in actual data (accounting for slight naming differences if any)
        actual_key = next((k for k in sections.keys() if sec_key in k), sec_key)
        item_count = len(sections.get(actual_key, []))

        # Only render line if it's the last item
        is_last = (idx == len(section_config))
        tree_branch = "└──" if is_last else "├──"

        html += f"""
                <li style="position: relative;">
                    <span style="color: #94a3b8; margin-right: 5px;">{tree_branch}</span>
                    {idx}. <a href="#{sec_key}" class="tree-link">{sec_key}</a>
                    <span class="tree-count">[{item_count}]</span>
                </li>
        """
    html += """
            </ul>
        </div>
        <div class="content-body">
    """

    # 3. Build Detailed Sections
    for sec_key, config in section_config.items():
        actual_key = next((k for k in sections.keys() if sec_key in k), None)

        if not actual_key or not sections[actual_key]:
            continue # Skip empty sections

        content_items = sections[actual_key]

        html += f"""
            <div class="section-box" id="{sec_key}">
                <div class="section-head">
                    <span>{config['icon']} {config['label']}</span>
                    <span class="badge-count">{len(content_items)} items</span>
                </div>
                <div class="section-content">
        """

        # Render internal blocks
        for item in content_items:
            if 'text' in item:
                html += f'<div class="block-text">{item["text"]}</div>'
            elif 'list' in item:
                html += '<div class="block-list"><ul>'
                for li in item['list']:
                    html += f'<li>{li}</li>'
                html += '</ul></div>'
            elif 'table' in item:
                # Basic table render (can use your previous complex table CSS if preferred)
                html += '<div class="block-text"><i>[Table Data: Use standard table rendering here]</i></div>'

        html += """
                </div>
            </div>
        """

    html += """
        </div>
    </div>
    """

    return HTML(html)

# ==========================================
# HOW TO EXECUTE IT
# ==========================================

# Example: Assuming 'dynamic_data' is your loaded JSON list
target_source_url = "https://www.physio-pedia.com/Anterior_Cruciate_Ligament_(ACL)_Injury" # Replace with your target HTML unique key

print(f"\n🔍 Fetching specialized view for: {target_source_url}")
display(display_injury_by_source(dynamic_data, target_source_url))

In [ ]:

dynamic_data=josn.load(open(""))

In [ ]:
def display_unique_urls(data):
    unique_urls = set()

    for injury in data:
        url = injury.get('source_url')
        if url:  # Make sure the URL actually exists and isn't empty
            unique_urls.add(url)

    print(f"Found {len(unique_urls)} unique source URLs:\n")
    print("-" * 60)

    for url in sorted(unique_urls):
        print(url)

    print("-" * 60)

display_unique_urls(dynamic_data)